In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

In [ ]:
embeddings_df=pd.read_csv(
    "/Users/007vd/Downloads/DAU/grl-paper/data/embeddings/graph_embeddings.csv"
)

embeddings_df

In [ ]:
embeddings_df.columns

In [ ]:
print(f"shape :{embeddings_df.shape}")
print("tickers: ")
print(embeddings_df["ticker"].unique())
print("Date Range:")

print(
    embeddings_df["date"].min(),
    "→",
    embeddings_df["date"].max()
)

In [ ]:
embedding_cols=[]
for col in embeddings_df.columns:
    if col.__contains__("emb_"):
        embedding_cols.append(col)
embedding_cols

In [ ]:
if "target" in embeddings_df.columns:
    embeddings_df=embeddings_df.drop(columns=["target"])


In [ ]:
embeddings_df["date"]=pd.to_datetime(embeddings_df["date"])
embeddings_df=embeddings_df.sort_values(["date","ticker"]).reset_index(drop=True)
embeddings_df

In [ ]:
embedding_stats = embeddings_df[embedding_cols].describe()

print(embedding_stats)

In [ ]:
embedding_variance = embeddings_df[embedding_cols].var().sort_values(
    ascending=False
)

print(embedding_variance)

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    embedding_variance.index,
    embedding_variance.values
)

plt.xticks(rotation=45)

plt.title("Variance Across Embedding Dimensions")

plt.xlabel("Embedding Dimension")

plt.ylabel("Variance")

plt.show()

In [ ]:
aapl_embeddings=embeddings_df[embeddings_df["ticker"]=="AAPL"].copy()
aapl_embeddings


In [ ]:
plt.figure(figsize=(14, 6))

for col in embedding_cols[:4]:

    plt.plot(
        aapl_embeddings["date"],
        aapl_embeddings[col],
        label=col
    )

plt.title(
    "Temporal Evolution of AAPL Embeddings"
)

plt.xlabel("Date")

plt.ylabel("Embedding Value")

plt.legend()

plt.show()

In [ ]:
scaler=StandardScaler()
scaled_embeddings=scaler.fit_transform(embeddings_df[embedding_cols])
scaled_embeddings

In [ ]:
pca=PCA(n_components=2)
pca_embeddings=pca.fit_transform(scaled_embeddings)
print(pca.explained_variance_ratio_)

In [ ]:
pca_df = pd.DataFrame({

    "pca_1": pca_embeddings[:, 0],

    "pca_2": pca_embeddings[:, 1],

    "ticker": embeddings_df["ticker"],

    "date": embeddings_df["date"]
})

pca_df

In [ ]:
plt.figure(figsize=(12, 8))

tickers = pca_df["ticker"].unique()

for ticker in tickers:

    subset = pca_df[
        pca_df["ticker"] == ticker
    ]

    plt.scatter(

        subset["pca_1"],

        subset["pca_2"],

        s=10,

        alpha=0.5,

        label=ticker
    )

plt.title(
    "PCA Projection of Graph Embeddings"
)

plt.xlabel("PCA Component 1")

plt.ylabel("PCA Component 2")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.show()

In [ ]:
aapl_pca = pca_df[
    pca_df["ticker"] == "AAPL"
].copy()

plt.figure(figsize=(12, 8))

scatter = plt.scatter(

    aapl_pca["pca_1"],

    aapl_pca["pca_2"],

    c=np.arange(len(aapl_pca)),

    cmap="viridis",

    s=12
)

plt.colorbar(
    scatter,
    label="Time Progression"
)

plt.title(
    "AAPL Trajectory Through PCA Embedding Space"
)

plt.xlabel("PCA 1")

plt.ylabel("PCA 2")

plt.show()